In [107]:
import pandas as pd

df = pd.read_excel("../clarified_dataset.xlsx")

In [108]:
df = df[df['th_cls_breast'] == 0]

#В первом приближении берём только температуры
x_cols = [f'ir_{i}' for i in range(0, 9)]
y_cols = [f'mw_{i}' for i in range(0, 9)]

In [109]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    df[x_cols], df[y_cols], test_size=0.2, random_state=42)

reg = LinearRegression()
reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)

mean_absolute_error(y_test, y_pred)

0.391840260052555

In [110]:
df[x_cols].iloc[0]

ir_0    33.0
ir_1    33.4
ir_2    32.7
ir_3    33.4
ir_4    32.6
ir_5    32.9
ir_6    32.8
ir_7    32.7
ir_8    33.0
Name: 0, dtype: float64

In [111]:
import numpy as np

def rotate(row, yrow):
    temps = np.array(row[1:])
    ytemps = np.array(yrow[1:])

    max_index = np.argmax(temps)

    if temps[max_index-1] < temps[(max_index+1) % len(temps)]: 
        temps =  np.roll(temps, -max_index)
        ytemps = np.roll(ytemps, -max_index)
    else:        
        temps =  np.roll(temps, -max_index)[::-1]
        ytemps = np.roll(ytemps, -max_index)[::-1]

    result =  np.insert(temps, 0, row.iloc[0]), np.insert(ytemps, 0, yrow.iloc[0])
    return pd.Series(result[0].tolist() + result[1].tolist())

df[x_cols+y_cols] = df[x_cols+y_cols].apply(lambda row: rotate(row[x_cols], row[y_cols]), axis=1)


In [112]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train, X_test, y_train, y_test = train_test_split(
    df[x_cols], df[y_cols], test_size=0.2, random_state=42)


X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_scaled = scaler.fit_transform(y_train)
y_test_scaled = scaler.transform(y_test)

reg = xgb.XGBRegressor(objective='reg:squarederror',
                         n_estimators=15, random_state=42)
reg.fit(X_train_scaled, y_train_scaled)

y_pred = reg.predict(X_test_scaled)

mean_absolute_error(scaler.inverse_transform(y_test_scaled), scaler.inverse_transform(y_pred))

0.3743239757735938

In [ ]:
import statsmodels.api as sm

X = df[x_cols]
X = sm.add_constant(X)
y = df['mw_0']

model = sm.OLS(y, X).fit()

print(model.summary())